## Demonstration: Using the Provenance Extension

**Prerequisite to run this notebook:** having [almond jupyter plugin](https://almond.sh/) installed, and running cells using a Scala kernel

To use Spark, we can use `$ivy` magic: https://almond.sh/docs/usage-spark

In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import $ivy.$

We can also use `ivy` to load project source code from local Ivy/Maven cache (run `make publish-local` to update):

In [2]:
val version = scala.io.Source.fromFile("../VERSION")  // Get version from file
  .getLines().next().trim

interp.load.ivy("org.dataprov.dp" %% "dp-spark" % version)  // use porogrammatic API 

// // For publishLocal (~/.ivy2/local)
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

// // For publishM2 (~/.m2)
// import $repo.`file:///home/ronan/.m2/repository`
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

version: String = "0.0.1"

Import libraries (Spark, dataprovenance project, ...):

In [3]:
import java.sql.Date

import org.apache.spark.sql.{SparkSession, DataFrame}
import org.apache.spark.sql.catalyst.plans.logical.LogicalPlan
import org.apache.spark.sql.execution.SparkPlan
import org.apache.spark.sql.functions._

import org.dataprov.dp.sparkdataprovenance.DataFrameProvenanceTransformations._
import org.dataprov.dp.LogicalPlanWithProvenance
import org.dataprov.dp.ProvenanceExtension
import org.dataprov.dp.WhyProvenanceBuilder

import java.sql.Date
import org.apache.spark.sql.{SparkSession, DataFrame}
import org.apache.spark.sql.catalyst.plans.logical.LogicalPlan
import org.apache.spark.sql.execution.SparkPlan
import org.apache.spark.sql.functions._
import org.dataprov.dp.sparkdataprovenance.DataFrameProvenanceTransformations._
import org.dataprov.dp.LogicalPlanWithProvenance
import org.dataprov.dp.ProvenanceExtension
import org.dataprov.dp.WhyProvenanceBuilder

The WhyProvenanceBuilder is a simple implementation of the ProvenanceBuilder trait that captures 
the logical plan and physical plan for each transformation. 

It can be used to track the lineage of data through the transformations and understand how the final result was derived from the input data.

## Initialize Spark session:
- The provenance builder can be customized to use different provenance models (e.g. WhyProvenanceBuilder) the default is DisplayStringProvenanceBuilder
- A new provenance builder can be created directly by overriding the operations (e.g. provType, single, join, distinct, aggregate)
- The name of the provenance column can also be customized the default is "_provenance_tag"

In [4]:
// Create SparkSession
val sparkWhy = SparkSession.builder()
    .appName("notebook-demo-why-provenance")
    .master("local[*]")
    .withExtensions(
        // can be customized with different operators and builders
        new ProvenanceExtension(provenanceBuilder = WhyProvenanceBuilder)
    )
    .config("spark.provenance.enabled", "true")
    .getOrCreate()

println(s"Spark provenance enabled: ${sparkWhy.conf.get("spark.provenance.enabled")}")

// Set log level to ERROR to reduce verbosity
sparkWhy.sparkContext.setLogLevel("ERROR")

// Set the name of the provenance column to "why_prov" 
// (optional, since the default is "_provenance_tag")
// sparkWhy.conf.set("spark.provenance.columnName", "why_prov")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/02 10:07:50 INFO SparkContext: Running Spark version 4.1.1
26/06/02 10:07:50 INFO SparkContext: OS info Mac OS X, 26.4.1, aarch64
26/06/02 10:07:50 INFO SparkContext: Java version 17.0.10+7
26/06/02 10:07:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/02 10:07:50 INFO ResourceUtils: ==============================================================
26/06/02 10:07:50 INFO ResourceUtils: No custom resources configured for spark.driver.
26/06/02 10:07:50 INFO ResourceUtils: ==============================================================
26/06/02 10:07:50 INFO SparkContext: Submitted application: notebook-demo-why-provenance
26/06/02 10:07:51 INFO SecurityManager: Changing view acls to: mac-ABALLA16
26/06/02 10:07:51 INFO SecurityManager: Changing modify acls to: mac-ABALLA16
26/06/02 10:07:51 INFO SecurityManager: Changing

Spark provenance enabled: true


sparkWhy: SparkSession = org.apache.spark.sql.classic.SparkSession@2667e3aa

Create Spark dataframe:

In [5]:
val df: DataFrame = sparkWhy.createDataFrame(
    Seq(
        ("A", Date.valueOf("2026-01-15"), 10.0, 90),
        ("A", Date.valueOf("2026-01-16"), 10.0, 120),
        ("A", Date.valueOf("2026-01-17"), 5.0, 300),
        ("B", Date.valueOf("2026-01-15"), 100.0, 20),
        ("B", Date.valueOf("2026-01-16"), 100.0, 30),
        ("C", Date.valueOf("2026-01-17"), 80.0, 60),
        ("C", Date.valueOf("2026-01-18"), 82.0, 50)
    )
).toDF("product", "date", "price", "sales")

df.show()

+-------+----------+-----+-----+
|product|      date|price|sales|
+-------+----------+-----+-----+
|      A|2026-01-15| 10.0|   90|
|      A|2026-01-16| 10.0|  120|
|      A|2026-01-17|  5.0|  300|
|      B|2026-01-15|100.0|   20|
|      B|2026-01-16|100.0|   30|
|      C|2026-01-17| 80.0|   60|
|      C|2026-01-18| 82.0|   50|
+-------+----------+-----+-----+



df: DataFrame = [product: string, date: date ... 2 more fields]

## Add provenance column to the DataFrame

The provenance column is added using the `addProvenanceColumn` method, which is provided by the `DataFrameProvenanceTransformations` trait. 

This method adds a new column to the DataFrame that contains the provenance information for each row. By default the provenance type is a uuid but can be customized by choosing a column name of the dataframe.

By adding the provenance column, users can easily trace back the origin of each row and understand the transformations that were applied to it, which can be useful for debugging and analysing purposes.


In [6]:
val dfWithProv : DataFrame = df.addProvenanceColumn
dfWithProv.show(false)

val dfWithProv2 : DataFrame = df.addProvenanceColumn(col("product"))
dfWithProv2.show(false)

+-------+----------+-----+-----+------------------------------------+
|product|date      |price|sales|_provenance_tag                     |
+-------+----------+-----+-----+------------------------------------+
|A      |2026-01-15|10.0 |90   |9d033e5d-7a44-4107-acec-721c028bd168|
|A      |2026-01-16|10.0 |120  |bd60db59-ed59-46c5-b61a-34751d85d15e|
|A      |2026-01-17|5.0  |300  |099dd746-649d-4416-b2e7-d5805e15896d|
|B      |2026-01-15|100.0|20   |09db289b-3e79-4009-9ded-516c71e441f4|
|B      |2026-01-16|100.0|30   |91a742cb-0f44-4e4a-9196-01ade973611c|
|C      |2026-01-17|80.0 |60   |217fcd47-b45d-41a4-adb1-eb3b5711afda|
|C      |2026-01-18|82.0 |50   |f439fad8-b403-486f-a862-2be68f091e31|
+-------+----------+-----+-----+------------------------------------+

+-------+----------+-----+-----+---------------+
|product|date      |price|sales|_provenance_tag|
+-------+----------+-----+-----+---------------+
|A      |2026-01-15|10.0 |90   |A              |
|A      |2026-01-16|10.0 |120  |A

dfWithProv: DataFrame = [product: string, date: date ... 3 more fields]
dfWithProv2: DataFrame = [product: string, date: date ... 3 more fields]

## Use the new DataFrame with provenance

You can use the same SQL or DataFrame API as before, but now you have access to the provenance information for debugging and analysis purposes.
You have nothing more to do, the provenance column will be calculated automatically.

Example query: find products with price > 50 and sales < 50.

In [7]:
// SQL API with provenance
dfWithProv.createOrReplaceTempView("why_prov")
val resultWithProv = sparkWhy.sql(
    """
    SELECT product, date, price, sales
    FROM why_prov
    WHERE price > 50 AND sales < 50
    """
)
resultWithProv.show(false)

// DataFrame API with provenance
val resultWithProv2 = dfWithProv
    .filter(col("price") > 50 && col("sales") < 50)
    .select("product", "date", "price", "sales")
resultWithProv2.show(false)

+-------+----------+-----+-----+------------------------------------+
|product|date      |price|sales|_provenance_tag                     |
+-------+----------+-----+-----+------------------------------------+
|B      |2026-01-15|100.0|20   |09db289b-3e79-4009-9ded-516c71e441f4|
|B      |2026-01-16|100.0|30   |91a742cb-0f44-4e4a-9196-01ade973611c|
+-------+----------+-----+-----+------------------------------------+

+-------+----------+-----+-----+------------------------------------+
|product|date      |price|sales|_provenance_tag                     |
+-------+----------+-----+-----+------------------------------------+
|B      |2026-01-15|100.0|20   |09db289b-3e79-4009-9ded-516c71e441f4|
|B      |2026-01-16|100.0|30   |91a742cb-0f44-4e4a-9196-01ade973611c|
+-------+----------+-----+-----+------------------------------------+



resultWithProv: DataFrame = [product: string, date: date ... 3 more fields]
resultWithProv2: DataFrame = [product: string, date: date ... 3 more fields]

### Test join with provenance

In [8]:
// left : filter on price and sales, price > 50 AND sales < 50
val left = dfWithProv.select("product", "date", "price", "sales").filter(col("price") > 50 && col("sales") < 50)
// right : filter on date, date > 2026-01-15
val right = dfWithProv.select("product", "date", "price", "sales").filter(col("date") > Date.valueOf("2026-01-15"))
// join on product, inner join, 
val joined = left.join(right, Seq("product"), "inner")

// show all three
println("Left:")
left.show(false)

println("Right:")
right.show(false)

println("Joined:")
joined.show(false)

Left:
+-------+----------+-----+-----+------------------------------------+
|product|date      |price|sales|_provenance_tag                     |
+-------+----------+-----+-----+------------------------------------+
|B      |2026-01-15|100.0|20   |09db289b-3e79-4009-9ded-516c71e441f4|
|B      |2026-01-16|100.0|30   |91a742cb-0f44-4e4a-9196-01ade973611c|
+-------+----------+-----+-----+------------------------------------+

Right:
+-------+----------+-----+-----+------------------------------------+
|product|date      |price|sales|_provenance_tag                     |
+-------+----------+-----+-----+------------------------------------+
|A      |2026-01-16|10.0 |120  |bd60db59-ed59-46c5-b61a-34751d85d15e|
|A      |2026-01-17|5.0  |300  |099dd746-649d-4416-b2e7-d5805e15896d|
|B      |2026-01-16|100.0|30   |91a742cb-0f44-4e4a-9196-01ade973611c|
|C      |2026-01-17|80.0 |60   |217fcd47-b45d-41a4-adb1-eb3b5711afda|
|C      |2026-01-18|82.0 |50   |f439fad8-b403-486f-a862-2be68f091e31|
+-----

left: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [product: string, date: date ... 3 more fields]
right: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [product: string, date: date ... 3 more fields]
joined: DataFrame = [product: string, date: date ... 6 more fields]

### Test distinct with provenance

In [ ]:

// Get distinct products from the joined result
// The provenance column will contain the provenance information, only one of 
val distinctProducts = dfWithProv.select("product").distinct()
distinctProducts.show(false)

+-------+--------------------------------------+
|product|_provenance_tag                       |
+-------+--------------------------------------+
|A      |[099dd746-649d-4416-b2e7-d5805e15896d]|
|B      |[91a742cb-0f44-4e4a-9196-01ade973611c]|
|C      |[f439fad8-b403-486f-a862-2be68f091e31]|
+-------+--------------------------------------+



distinctProducts: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [product: string, _provenance_tag: array<string>]

### Test aggregations with provenance

In [ ]:
val aggreg = dfWithProv.groupBy("product").agg(
    max("date").as("first_date"),
    min("date").as("last_date"),
    round(mean("price"), 2).as("average_price"),
    sum("sales").as("total_sales")
)
aggreg.show(false)

val aggreg2 = dfWithProv2.groupBy("product").agg(
    max("date").as("first_date"),
    min("date").as("last_date"),
    round(mean("price"), 2).as("average_price"),
    sum("sales").as("total_sales")
)
aggreg2.show(false)

+-------+----------+----------+-------------+-----------+------------------------------------------------------------------------------------------------------------------+
|product|first_date|last_date |average_price|total_sales|_provenance_tag                                                                                                   |
+-------+----------+----------+-------------+-----------+------------------------------------------------------------------------------------------------------------------+
|A      |2026-01-15|2026-01-17|8.33         |510        |[099dd746-649d-4416-b2e7-d5805e15896d, bd60db59-ed59-46c5-b61a-34751d85d15e, 9d033e5d-7a44-4107-acec-721c028bd168]|
|B      |2026-01-15|2026-01-16|100.0        |50         |[09db289b-3e79-4009-9ded-516c71e441f4, 91a742cb-0f44-4e4a-9196-01ade973611c]                                      |
|C      |2026-01-17|2026-01-18|81.0         |110        |[217fcd47-b45d-41a4-adb1-eb3b5711afda, f439fad8-b403-486f-a862-2be68f091e31]  

aggreg: DataFrame = [product: string, first_date: date ... 4 more fields]
aggreg2: DataFrame = [product: string, first_date: date ... 4 more fields]